In [1]:
import torch, tqdm, time, os, json
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from models.est_model import GroundPressureModel
from torchmetrics import Accuracy
from torch.utils.data import DataLoader, Dataset, random_split
from pytorch_lightning.callbacks import DeviceStatsMonitor, EarlyStopping
from sklearn.metrics import r2_score

In [2]:
monitor = DeviceStatsMonitor()

In [3]:
# set train setting
save_model = True
train_num_epoch = 50000
min_loss = 100
dl_workers = 0

In [4]:
gpu = torch.device('cuda')
torch.set_float32_matmul_precision('high')

In [5]:
# load hyperparameter of json file
with open('.' + os.sep + os.path.join('models', 'params_dnn_20220207-012403.json'), 'r') as file:
    hyper_params = json.load(file)

n_input = hyper_params['n_of_inputs']
h1_neuron = hyper_params['h1_neuron']
h2_neuron = hyper_params['h2_neuron']
n_output = hyper_params['n_of_outputs']

In [6]:
model = GroundPressureModel(n_input, h1_neuron, h2_neuron, n_output)
print(model)

GroundPressureModel(
  (model): Sequential(
    (0): Linear(in_features=4, out_features=24, bias=True)
    (1): ReLU()
    (2): Linear(in_features=24, out_features=24, bias=True)
    (3): ReLU()
    (4): Linear(in_features=24, out_features=24, bias=True)
    (5): ReLU()
    (6): Linear(in_features=24, out_features=24, bias=True)
    (7): ReLU()
    (8): Linear(in_features=24, out_features=6, bias=True)
  )
  (loss_func): MSELoss()
)


In [7]:
data = pd.read_csv('./resources/sim_data.csv')
feature_names = ['lift_weight(ton)', 'lift_height(m)', 'rising_angle(deg)', 'swing_angle(deg)']
target_names = ['left_ground_pressure_min(kg/cm2)', 'left_ground_pressure_max(kg/cm2)', 'left_pressure_length(m)', 'right_ground_pressure_min(kg/cm2)', 'right_ground_pressure_max(kg/cm2)', 'right_pressure_length(m)']

feature_data = []
target_data = []

for feature_name in feature_names:
    feature_data.append(data[feature_name])

for target_name in target_names:
    target_data.append(data[target_name])

feature_data = np.array(feature_data, dtype=np.float32).T
target_data = np.array(target_data, dtype=np.float32).T

class TensorDataset(Dataset):
    def __init__(self, feature, target):
        self.x_data = torch.tensor(feature)
        self.y_data = torch.tensor(target)
        self.len = self.x_data.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len

train_dataset = TensorDataset(feature_data, target_data)
train_data_loader = DataLoader(train_dataset, batch_size=len(train_dataset), num_workers=20)

/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/torch/utils/data/dataloader.py:561: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 16, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [8]:
# train model
early_stop_callback = EarlyStopping(monitor='train_loss', mode='min', verbose=True, min_delta=0.0001, patience=20)
trainer = pl.Trainer(callbacks=[early_stop_callback], accelerator='gpu', enable_progress_bar=True)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [9]:
trainer.fit(model=model, train_dataloaders=train_data_loader)

/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/pytorch_lightning/loops/utilities.py:70: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/pytorch_lightning/trainer/configuration_validator.py:72: PossibleUserWarning: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
  rank_zero_warn(
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type       | Params
-----------------------------------------
0 | model     | Sequential | 870   
1 | loss_func | MSELoss    | 0     
-----------------------------------------
870       Trainable params
0         Non-trainable params
870       Total params
0.003     Total estimated model params size (MB)
/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3

Epoch 0: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s, v_num=8]

Metric train_loss improved. New best score: 15.493


Epoch 1: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s, v_num=8]

Metric train_loss improved by 3.201 >= min_delta = 0.0001. New best score: 12.291


...

Epoch 526: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s, v_num=8]

Metric train_loss improved by 0.000 >= min_delta = 0.0001. New best score: 0.011


Epoch 546: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s, v_num=8]

Monitored metric train_loss did not improve in the last 20 records. Best score: 0.011. Signaling Trainer to stop.


Epoch 546: 100%|██████████| 1/1 [00:00<00:00,  1.90it/s, v_num=8]


In [11]:
torch.save(model.state_dict(), './models/est_ground_pressure.pt')

In [10]:
model.eval()
for data in train_dataset:
    print(r2_score(data[1].detach().numpy() ,model(data[0]).detach().numpy()))
    print(model(data[0]).detach().numpy())
    print(data[1].detach().numpy())

0.9965712991459258
[3.8621328e+00 1.9575769e-02 3.5686564e+00 3.9271710e+00 1.7584339e-03
 3.5438180e+00]
[3.97 0.   3.72 3.97 0.   3.72]
0.9969584650248057
[2.449417   0.00698435 3.94781    4.6006956  0.01534314 3.9694269 ]
[2.61 0.   4.1  4.58 0.   4.1 ]
0.9993749172326493
[1.0426897  0.01237076 5.635482   4.222221   0.00773941 5.5951457 ]
[1.15 0.   5.59 4.13 0.   5.59]
0.9960897495967124
[2.6531091e+00 2.1135751e-03 5.1332498e+00 2.8721306e+00 1.5043765e-03
 5.1471601e+00]
[2.93 0.   5.04 2.93 0.   5.04]
0.9996441717855639
[2.0827687  0.01356688 5.2926917  3.3548732  0.02862997 5.305738  ]
[2.16 0.   5.34 3.36 0.   5.34]
0.9994705878319287
[1.193835   0.04987683 6.4997077  3.1798933  0.09522825 6.5050964 ]
[1.25 0.   6.53 3.27 0.   6.53]
0.9992578064270785
[ 2.1944044  -0.00772533  6.5487776   2.2934139   0.01353411  6.5814157 ]
[2.21 0.   6.68 2.21 0.   6.68]
0.9992129334335428
[1.8236184  0.01856505 6.6103244  2.4797926  0.04802635 6.620456  ]
[1.83 0.04 6.75 2.46 0.05 6.75]
0.99